# Klassifikation - Mindestanforderungen_6

#### Führen Sie mit dem Algorithmus Ihrer Wahl eine Klassifikationsaufgabe auf Ihren Daten durch.
- Durchführung Klassifikation mit XGBoost (`XGBClassifier`)
- Zielvariable: `JobSatisfaction`
- Ursprüngliche Skala 0-10 zu 3 Klassen zusammengefasst
    - Low: 0-3
    - Medium: 4-6
    - High: 7-10
---
#### Teilen Sie dazu zunächst die Daten auf, um Overfitting beim Trainieren des Algorithmus und bei der Parameterauswahl zu vermeiden. Erklären Sie die gewählte Strategie und die Größenverhältnisse.
- **Gewählte Strategie**
    - **Stufe 1 (Modellwahl/Parametertuning)**
        - Parameter ausschließlich auf den Trainingsdaten optimiert –> dafür **3-fache Cross-Validation (cv=3)** im Trainingssplit
    - **Stufe 2 (Finale Bewertung)**
        - nachdem besten Parameter feststehen, wird finales Modell auf separaten Testsplit bewertet
    - durch `stratify=y` wird die Klassenverteilung in den Splits stabil gehalten

- **Konkrete Umsetzung im Code**
  - `train_test_split(..., test_size=0.2, stratify=y, random_state=42)` → **80% Train / 20% Test**
  - `GridSearchCV(..., cv=3)` → Cross-Validation innerhalb der **80% Trainingsdaten**

- **Größenverhältnisse:**
  - **Trainingsdaten/Testdaten:** 80% / 20%
---
#### Wählen Sie geeignete Features aus und setzen Sie die Parameter des Algorithmus. Beschreiben Sie das gewälhte Vorgehen für die Auswahl der Features und Parameter. Berichten Sie den Parameterraum und die final gewählten Parameter. Geben Sie die Performanz auf den Trainingsdaten (bzw. Entwicklungsdaten, falls verwendet) an.

- **Features**
    - Datengrundlage ist der One-Hot-Encodede Datensatz
    - `bool`-Werte wurden in 0/1 umgewandelt
    - entfernte Spalten
        - `JobSatisfaction`-Spalte, um Data Leakage zu vermeiden
        - IDs und weitere nicht informative Variablen
        - `ConvertedCompTotal` - hatte viele Missing Values und keinen Fokus dieses Modells
    - Fehlende Werte in numerischen Features werden mit Median-Imputation aufgefüllt
- **Vorgehen**
    - Pipeline mit `ColumnTransformer` (MedianImputation) und XGBClassifier
    - Hyperparameter-Optimierung mit `GridSearchCV` (`cv=3`) auf Trainingssplit
    - Optimierungsmaß: macro-F1, da jede Klasse gleich gewichtet wird und Klassen unbalanciert sind
- **Parameterraum (GridSearch) - XGBoost (`XGBClassifier`)**
    - `n_estimators`: 300, 500
    - `max_depth`: 4, 6
    - `learning_rate`: 0.05, 0.1
    - `reg_lambda`: 1.0, 2.0
    - `subsample`: 0.8
    - `colsample_bytree`: 0.8
- **Finale Parameter**
    - `n_estimators`: 500
    - `max_depth`: 4
    - `learning_rate`: 0.1
    - `reg_lambda`:1.0
    - `subsample`: 0.8
    - `colsample_bytree`: 0.8
- **Bestes Ergebnis aus Trainingsdaten**
    - 0.377
---
#### Evaluieren Sie die Klassifikation auf den ungesehenen Testdaten. Betrachten Sie Precision und Recall sowie den F-Wert. Welches Maß ist für Ihre Anwendung wichtiger? Bewerten Sie Ihr Ergebnis. Ist es in der Praxis voraussichtlich zufriedenstellend?

- **Test-Performance**
    - Accuracy: 0.70
    - weighted F1: 0.64
    - macro F1: 0.36
- **Klassenweise (Precision / Recall / F1)**
  - Low: 0.24 / 0.03 / 0.05
  - Medium: 0.34 / 0.14 / 0.20
  - High: 0.74 / 0.93 / 0.83
- **Interpretation**
    - Modell erkennt die Mehrheitsklasse High sehr zuverlässig (Recall 0.93, F1 0.83)
    - Low und Medium werden dagegen häufig fälschlich als High vorhergesagt
    - dadurch wirken Accuracy und weighted F1 relativ hoch, während die macro-F1 (0.36) niedrig bleibt, weil die Minderheitsklassen schlecht erkannt werden
- **Wichtigstes Maß**
    - macro F1 (da Klassen stark unausgewogen sind; Accuracy/weighted F1 werden von High dominiert)
- **Praxisbewertung**
    - nur bedingt zufriedenstellend, wenn alle drei Klassen gleich gut erkannt werden sollen
    - zufriedenstellend nur, wenn primär Klasse High erkannt werden soll


- prüft, in welchem Ordner Notebook sich befindet und ob in dem Ordner Dateien liegen, die wie xgboost* heißen
- lediglich fürs Debugging

In [15]:
import glob
import os

print("cwd:", os.getcwd())
print("local xgboost candidates:", glob.glob("xgboost*"))


cwd: C:\Users\MoritzSchwarz\PycharmProjects\data-analytics-project\Abgaben
local xgboost candidates: []


- Deinstallation bestehender XG-Boost Installation und anschließende Neuinstallation

In [16]:

!{sys.executable} -m pip -V
!{sys.executable} -m pip uninstall -y xgboost
!{sys.executable} -m pip install --no-cache-dir -U xgboost


pip 25.2 from C:\Users\MoritzSchwarz\PycharmProjects\data-analytics-project\.venv\Lib\site-packages\pip (python 3.13)

Found existing installation: xgboost 3.1.2
Uninstalling xgboost-3.1.2:
  Successfully uninstalled xgboost-3.1.2


You can safely remove it manually.


   ---------------------------------------- 0.0/72.0 MB ? eta -:--:--
   -- ------------------------------------- 5.0/72.0 MB 28.4 MB/s eta 0:00:03
   ------ --------------------------------- 11.0/72.0 MB 28.8 MB/s eta 0:00:03
   ---------- ----------------------------- 19.4/72.0 MB 32.7 MB/s eta 0:00:02
   -------------- ------------------------- 27.0/72.0 MB 34.0 MB/s eta 0:00:02
   ------------------ --------------------- 33.6/72.0 MB 33.8 MB/s eta 0:00:02
   ----------------------- ---------------- 41.7/72.0 MB 34.6 MB/s eta 0:00:01
   --------------------------- ------------ 50.1/72.0 MB 35.5 MB/s eta 0:00:01
   ------------------------------- -------- 57.4/72.0 MB 35.5 MB/s eta 0:00:01
   ------------------------------------ --- 65.8/72.0 MB 36.0 MB/s eta 0:00:01
   ---------------------------------------  71.3/72.0 MB 35.1 MB/s eta 0:00:01
   ---------------------------------------- 72.0/72.0 MB 35.2 MB/s  0:00:02



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.impute import SimpleImputer

## Daten laden

Im One-Hot-Encoded Datensatz sind die One-Hot-Encoded Spalten mit bool Werten. Um es etwas einfacher zu gestalten werden diese Werte hier in Integer-Werte (False -> 0; True -> 1) umgewandelt.

In [18]:
df = pd.read_csv("One-Hot-Encoded-final.csv")

bool_cols = df.select_dtypes(include=["bool"]).columns.tolist()
for c in bool_cols:
    df[c] = df[c].astype(int)

## Zielvariable in Klassen einteilen

Dadurch haben wir mehr Trainingsdaten, als wenn wir `JobSatisfaction` von 0-10 als Klassen definieren würden, `NaN`-Werte werden zuvor gedroppt

Klassen
- **Low**: 0–3
- **Medium**: 4–6
- **High**: 7–10


In [19]:
df = df.dropna(subset=["JobSatisfaction"]).copy()

def map_JobSatisfaction(x):
    x = float(x)
    if x <= 3:
        return 0  # Low
    elif x <= 6:
        return 1  # Medium
    else:
        return 2  # High

y = df["JobSatisfaction"].apply(map_JobSatisfaction)

## Feature-Spalten bestimmen

- Datengrundlage ist der One-Hot-Encoded Datensatz, daher liegen die Features bereits numerisch vor (bool wurde zu 0/1 umgewandelt).
- Zielvariable `JobSatisfaction` und nicht informative Spalten (z.B. `ResponseId`) werden aus den Features entfernt
- Falls noch `object`-Spalten vorhanden sind, werden diese entfernt.

In [20]:
# nicht benötigte Feature-Liste die gedroppt wird
drop_cols = [
    "JobSatisfaction",
    "ConvertedCompTotal",
    "ResponseId",
    "AgeNum"
]

X = df.drop(columns=drop_cols, errors="ignore")

obj_cols = X.select_dtypes(include=["object"]).columns.tolist()
if obj_cols:
    print("Noch object-Spalten gefunden und entfernt:", obj_cols)
    X = X.drop(columns=obj_cols)

num_cols = X.select_dtypes(include=["number"]).columns.tolist()
X = X[num_cols].copy()

print("X shape:", X.shape, "| y shape:", y.shape)
print("Class distribution:\n", y.value_counts())

X shape: (16624, 383) | y shape: (16624,)
Class distribution:
 JobSatisfaction
2    11992
1     3690
0      942
Name: count, dtype: int64


## Train/Test Split
- Aufteilen der Daten in 80% Training und 20% Test
- `stratify=y` sorgt für ungefähre Gleichverteilung der Klassen in Test und Trainingsdaten


In [21]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))
print("Train class distribution:\n", y_train.value_counts(normalize=True))
print("Test class distribution:\n", y_test.value_counts(normalize=True))

Train size: 13299
Test size: 3325
Train class distribution:
 JobSatisfaction
2    0.721332
1    0.221972
0    0.056696
Name: proportion, dtype: float64
Test class distribution:
 JobSatisfaction
2    0.721504
1    0.221955
0    0.056541
Name: proportion, dtype: float64


## Preprocessing
- alle Features numerisch
- fehlende Werte durch Median-Imputation ersetzt

In [22]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
        ]), num_cols)
    ],
    remainder="drop"
)

## Pipeline definieren

- Pipeline kombiniert Preprocessing (Median-Imputation) und den Klassifikator `XGBClassifier`.

In [23]:
xgb = XGBClassifier(
    objective="multi:softprob",
    eval_metric="mlogloss",
    n_jobs=-1,
    random_state=42
)

In [24]:
pipe = Pipeline([
    ("preprocessing", preprocessor),
    ("clf", xgb)
])

## GridSearchCV
- definiert Suchraum für Parameter
- GridSearch testet Kombinationen der Parameter systematisch und wählt die beste Kombination per Cross-Validation (`cv`)
- `cv=3` bedeutet, dass eine 3-fache Validierung auf dem Trainingsset vorgenommen wurde


In [25]:
param_grid = {
    "clf__n_estimators": [300, 500],
    "clf__max_depth": [4, 6],
    "clf__learning_rate": [0.05, 0.1],
    "clf__subsample": [0.8],
    "clf__colsample_bytree": [0.8],
    "clf__reg_lambda": [1.0, 2.0],
}
grid = GridSearchCV(
    pipe,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=3,
    n_jobs=-1,
    verbose=2
)

## Grid Search + Beste Parameter
- Viele Modellvarianten werden trainiert und die beste Variante gefunden
- Output ist dann der beste CV-Score mit den dazugehörigen Parametern

In [26]:
grid.fit(X_train, y_train)

print("\nBest CV macro-F1:", grid.best_score_)
print("Best params:", grid.best_params_)

Fitting 3 folds for each of 16 candidates, totalling 48 fits

Best CV macro-F1: 0.3777097555256974
Best params: {'clf__colsample_bytree': 0.8, 'clf__learning_rate': 0.1, 'clf__max_depth': 4, 'clf__n_estimators': 500, 'clf__reg_lambda': 1.0, 'clf__subsample': 0.8}


## Evaluation auf Testdaten
- Test-Datensatz Vorhersage samt Metriken
- Am Ende zählt Performance auf Test-Daten, welche Modell noch nicht kennt

In [27]:
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print("\nClassification Report (Test):")
print(classification_report(y_test, y_pred, zero_division=0))
print("Confusion Matrix (rows=true, cols=pred):")
print(confusion_matrix(y_test, y_pred, labels=[0, 1, 2]))


Classification Report (Test):
              precision    recall  f1-score   support

           0       0.24      0.03      0.05       188
           1       0.34      0.14      0.20       738
           2       0.74      0.93      0.83      2399

    accuracy                           0.70      3325
   macro avg       0.44      0.36      0.36      3325
weighted avg       0.62      0.70      0.64      3325

Confusion Matrix (rows=true, cols=pred):
[[   5   44  139]
 [   4  102  632]
 [  12  157 2230]]


- Ausgabe gleicher Metriken für Trainingsdaten um Test- und Trainingsdaten zu vergleichen
- Over- / Underfitting?

In [28]:
y_pred_train = best_model.predict(X_train)

print("\nClassification Report (Train):")
print(classification_report(y_train, y_pred_train, zero_division=0))


Classification Report (Train):
              precision    recall  f1-score   support

           0       0.98      0.49      0.65       754
           1       0.90      0.41      0.57      2952
           2       0.82      0.99      0.90      9593

    accuracy                           0.83     13299
   macro avg       0.90      0.63      0.71     13299
weighted avg       0.85      0.83      0.81     13299

